# Semana 7 · Cazando datos: el universo de los datos
**Fundamentos de Big Data — Taller práctico con Python y pandas**

> *¿Dónde están los datos y qué forma tienen?*

**Lectura base:** Joyanes Aguilar, L. *Big Data: Análisis de grandes volúmenes de datos en organizaciones*. Alfaomega. **Capítulo 3.**

## Objetivos de aprendizaje

- Reconocer las **fuentes** de datos de una organización según la clasificación del capítulo 3.
- Trabajar con datos **estructurados**, **semiestructurados** y **no estructurados** en Python.
- Experimentar en la práctica los límites del **volumen** y entender por qué existen las herramientas de Big Data.
- Construir un **inventario de fuentes** y valorar cuáles aportan más al negocio.


## Repaso: las fuentes de datos según Joyanes

El capítulo 3 organiza el universo de los datos en grandes categorías de fuentes. Complete la última columna con ejemplos **de Olist o de una empresa colombiana** que usted conozca.

| Categoría | Qué incluye | Ejemplo en una empresa |
|---|---|---|
| Web y redes sociales | Clics, navegación, búsquedas, publicaciones y comentarios en redes | |
| Máquina a máquina (M2M) | Sensores, GPS, medidores, dispositivos IoT que se comunican entre sí | |
| Grandes transacciones | Facturación, pagos, pedidos, registros de llamadas | |
| Biométricos | Huellas, reconocimiento facial, datos genéticos o médicos | |
| Generados por personas | Correos, documentos, notas, grabaciones de voz, encuestas | |

*Nota: verifique los nombres exactos de las categorías con la edición del libro que usa el curso.*

Además, según su **estructura**, los datos pueden ser:

- **Estructurados:** filas y columnas con un esquema fijo (tablas, bases de datos relacionales).
- **Semiestructurados:** tienen etiquetas u organización, pero su esquema es flexible (JSON, XML, registros de servidores).
- **No estructurados:** sin esquema predefinido (texto libre, imágenes, audio, video).

En este taller vamos a **cazar** una fuente de cada tipo.


### Cómo trabajar con este notebook

1. Si lo abrió en Google Colab: **Archivo → Guardar una copia en Drive** antes de empezar.
2. Ejecute las celdas en orden con **Shift + Enter**.
3. Las celdas marcadas con **COMPLETE AQUÍ** tienen espacios `___` que usted debe reemplazar por el código correcto. Cuando el espacio va entre comillas, escriba el **nombre exacto de la columna** tal como aparece en la tabla (por ejemplo `"order_id"`), no una descripción en español. Si escribe un nombre que no existe, pandas mostrará un error `KeyError`.
4. Las preguntas numeradas (**Pregunta 1.1**, **Pregunta 2.1**…) se responden en la celda de texto que aparece debajo (doble clic para editarla).
5. Al final de cada sección hay una tabla **Documentación para consultar** con enlaces a la documentación oficial de cada término usado (por ejemplo `.dt`, `merge` o `groupby`). Al final del notebook están todos reunidos. También puede ver la ayuda sin salir del notebook escribiendo `help(pd.merge)` o `pd.merge?`.


## 0. Preparar el entorno

In [ ]:
import os, json, re, time
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)
plt.rcParams["figure.figsize"] = (10, 4)

### Carga de los datos

Usamos el **Brazilian E-Commerce Public Dataset by Olist** ([Kaggle](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)).

La celda siguiente busca los archivos en este orden:

1. **Carpeta `datos/`** junto al notebook (por ejemplo, si subió los CSV a Colab con el ícono de carpeta o si clonó el repositorio).
2. **Google Drive**, en `Mi unidad/olist/` (si el docente compartió la carpeta; primero monte Drive con el ícono de carpeta → *Montar Drive*).
3. **Descarga directa desde Kaggle** con `kagglehub`. Si le pide credenciales, use la opción 1 o 2.


In [ ]:
import os

def obtener_carpeta_datos():
    """Devuelve la carpeta donde están los CSV de Olist."""
    candidatas = ["datos", "../datos", "/content/datos", "/content/drive/MyDrive/olist"]
    for carpeta in candidatas:
        if os.path.exists(os.path.join(carpeta, "olist_orders_dataset.csv")):
            print("Datos encontrados en:", carpeta)
            return carpeta
    print("Descargando desde Kaggle (puede tardar un minuto)...")
    try:
        import kagglehub
    except ImportError:
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"], check=True)
        import kagglehub
    carpeta = kagglehub.dataset_download("olistbr/brazilian-ecommerce")
    print("Datos descargados en:", carpeta)
    return carpeta

CARPETA = obtener_carpeta_datos()
sorted(f for f in os.listdir(CARPETA) if f.endswith(".csv"))

---
## 1. Fuente 1 · Datos estructurados: las transacciones

La tabla de pedidos es el ejemplo clásico de **dato estructurado** proveniente de **transacciones**: cada fila es un pedido y cada columna tiene un tipo definido.

In [ ]:
pedidos = pd.read_csv(f"{CARPETA}/olist_orders_dataset.csv",
                      parse_dates=["order_purchase_timestamp", "order_delivered_customer_date",
                                   "order_estimated_delivery_date"])
pedidos.info()

### Recordatorio: cómo ver las columnas

Para completar los ejercicios necesita los **nombres exactos** de las columnas. Úselos tal como aparecen, entre comillas.

| Instrucción | Qué muestra |
|---|---|
| `tabla.columns.tolist()` | La lista de nombres de columnas |
| `tabla.head()` | Las primeras filas, con los nombres en el encabezado |
| `tabla.info()` | Cada columna con su tipo de dato y cuántos valores no están vacíos (como la celda anterior) |

**Columnas de Olist que se usan hoy**

| Tabla | Columna | Significado |
|---|---|---|
| pedidos | `order_status` | Estado del pedido (entregado, enviado, cancelado…) |
| pedidos | `order_delivered_customer_date` | Fecha real de entrega al cliente |
| pedidos | `order_estimated_delivery_date` | Fecha estimada de entrega |
| resenas | `review_score` | Calificación de 1 a 5 estrellas |
| resenas | `review_comment_message` | Comentario escrito por el cliente, en portugués |
| geo | `geolocation_zip_code_prefix` | Primeros dígitos del código postal |
| geo | `geolocation_lat`, `geolocation_lng` | Latitud y longitud |
| geo | `geolocation_state` | Estado (departamento) de Brasil |

Las columnas de SECOP II se muestran más adelante, cuando se descarguen.

**Veracidad:** un dato estructurado no es necesariamente un dato completo. Revisemos los valores faltantes.

In [ ]:
nulos = pd.DataFrame({
    "faltantes": pedidos.isna().sum(),
    "% faltante": pedidos.isna().mean().mul(100).round(2),
})
nulos[nulos["faltantes"] > 0]

**COMPLETE AQUÍ**: cuente cuántos pedidos hay en cada estado (`order_status`). Pista: el método que cuenta valores repetidos se llama `value_counts`.

In [ ]:
# COMPLETE AQUÍ: reemplace cada ___
pedidos["order_status"].___()

#### Documentación para consultar

Términos usados en esta sección. Consulte los enlaces para ampliar.

| Término | Para qué sirve | Enlace |
|---|---|---|
| `pd.read_csv()` | Lee un archivo CSV y lo convierte en tabla (DataFrame). `parse_dates` indica qué columnas son fechas y `chunksize` permite leer por bloques | [Ver documentación](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) |
| `.info()` | Resumen de la tabla: columnas, tipos de dato y valores no vacíos | [Ver documentación](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.info.html) |
| `.isna()` | Devuelve `True` donde el valor está vacío (NaN) | [Ver documentación](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.isna.html) |
| `.mean()` | Promedio. Sobre una columna de `True`/`False` da la proporción de `True` | [Ver documentación](https://pandas.pydata.org/docs/reference/api/pandas.Series.mean.html) |
| `.mul()` / `.round()` | Multiplica (por ejemplo, por 100 para obtener porcentajes) y redondea | [Ver documentación](https://pandas.pydata.org/docs/reference/api/pandas.Series.mul.html) |
| `.value_counts()` | Cuenta cuántas veces aparece cada valor. Con `normalize=True` da proporciones | [Ver documentación](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html) |
| Guía: datos faltantes | Cómo detectar y tratar valores vacíos | [Ver documentación](https://pandas.pydata.org/docs/user_guide/missing_data.html) |

**Pregunta 1.1.** ¿Por qué hay pedidos sin fecha de entrega? ¿Es un error del sistema o refleja algo real del negocio? ¿Qué V de Big Data se relaciona con esto?

*Respuesta:* 

---
## 2. Fuente 2 · Datos semiestructurados: datos abiertos del Estado colombiano

Muchas organizaciones publican datos a través de una **API** (interfaz de programación): una dirección web que, en lugar de una página, devuelve datos en formato **JSON**.

Vamos a consultar **SECOP II – Contratos Electrónicos** en [datos.gov.co](https://www.datos.gov.co/Gastos-Gubernamentales/SECOP-II-Contratos-Electr-nicos/jbjy-vk9h), la plataforma de datos abiertos de Colombia. Es una fuente **externa** para cualquier empresa que venda al Estado.

El parámetro `$limit` indica cuántos registros traer. El conjunto completo tiene millones de contratos; aquí traemos una muestra.

In [ ]:
import requests

ID_DATASET = "jbjy-vk9h"          # SECOP II - Contratos Electrónicos
URL = f"https://www.datos.gov.co/resource/{ID_DATASET}.json"
parametros = {"$limit": 5000}

try:
    respuesta = requests.get(URL, params=parametros, timeout=120)
    respuesta.raise_for_status()
    registros = respuesta.json()
    with open("secop.json", "w", encoding="utf-8") as f:
        json.dump(registros, f, ensure_ascii=False)
    print(f"Descargados {len(registros):,} registros desde la API")
except Exception as error:
    print("No se pudo consultar la API:", error)
    if os.path.exists("secop.json"):
        with open("secop.json", encoding="utf-8") as f:
            registros = json.load(f)
        print(f"Se usará la copia local secop.json ({len(registros):,} registros)")
    else:
        raise RuntimeError("Sin conexión a la API y sin copia local. Pida al docente el archivo secop.json.")

Así se ve **un registro** tal como llega de la API (se muestran los primeros caracteres):

In [ ]:
print(json.dumps(registros[0], indent=2, ensure_ascii=False)[:1500])

Observe que el campo `urlproceso` no es un valor simple: es **otro objeto dentro del registro** (datos anidados). Eso no existe en una tabla CSV.

¿Todos los registros traen los mismos campos? En una tabla estructurada, sí. Veamos aquí:

In [ ]:
campos_por_registro = pd.Series([len(r) for r in registros])
print("Número de campos por registro:")
print(campos_por_registro.value_counts().sort_index().to_string())

todos_los_campos = set().union(*registros)
presencia = pd.Series({c: sum(c in r for r in registros) for c in todos_los_campos})
incompletos = presencia[presencia < len(registros)].sort_values()
print(f"\n{len(incompletos)} campos no aparecen en todos los registros")
incompletos.head(10)

Para analizarlo con pandas, **aplanamos** el JSON con `json_normalize`: los campos anidados se convierten en columnas como `urlproceso.url`, y los campos ausentes quedan como `NaN`.

In [ ]:
secop = pd.json_normalize(registros)
print("Filas y columnas:", secop.shape)
secop.filter(like="url").head(3)

Estas son **todas las columnas** que trae SECOP II. Busque aquí los nombres que necesite en los ejercicios.

In [ ]:
secop.columns.tolist()

**Columnas de SECOP II que se usan en el taller**

| Columna | Significado |
|---|---|
| `nombre_entidad` | Entidad pública que contrata |
| `departamento`, `ciudad` | Ubicación de la entidad |
| `tipo_de_contrato` | Prestación de servicios, suministro, obra… |
| `modalidad_de_contratacion` | Contratación directa, licitación pública, mínima cuantía… |
| `fecha_de_firma` | Fecha en que se firmó el contrato |
| `valor_del_contrato` | Valor del contrato en pesos colombianos |
| `proveedor_adjudicado` | Persona o empresa que ganó el contrato |
| `urlproceso.url` | Enlace al proceso en el portal SECOP (era un campo anidado en el JSON) |

La descripción oficial de cada columna aparece en la página del conjunto en datos.gov.co, en la sección **Columnas de esta tabla**.

In [ ]:
# La API entrega todo como texto: fíjese en los tipos de dato
secop[["valor_del_contrato", "fecha_de_firma", "departamento"]].dtypes

**COMPLETE AQUÍ**: convierta el valor del contrato a número. La columna se llama `valor_del_contrato`.

In [ ]:
# COMPLETE AQUÍ: reemplace cada ___
secop["valor_del_contrato"] = pd.to_numeric(secop["___"], errors="coerce")
secop["valor_del_contrato"].describe()

**COMPLETE AQUÍ**: sume el valor de los contratos por **departamento** y grafique los 10 mayores.

| Espacio | Qué escribir |
|---|---|
| Columna para agrupar | `departamento` |
| Método para sumar | `sum` |

In [ ]:
# COMPLETE AQUÍ: reemplace cada ___
por_depto = secop.groupby("___")["valor_del_contrato"].___().sort_values().tail(10)

(por_depto / 1e9).plot(kind="barh", title="Valor contratado por departamento (muestra)")
plt.xlabel("Miles de millones de pesos")
plt.show()

### Datos personales y ética

Aunque sean datos públicos, algunas columnas identifican a **personas naturales**. En Colombia, la **Ley 1581 de 2012** regula el tratamiento de datos personales.

In [ ]:
pistas = ["documento", "nombre", "identificaci", "cuenta", "domicilio", "g_nero"]
columnas_personales = [c for c in secop.columns
                       if any(p in c for p in pistas) and "entidad" not in c]
columnas_personales

**COMPLETE AQUÍ**: cree una versión anonimizada eliminando esas columnas. Pista: la lista de columnas quedó guardada en la variable `columnas_personales` (va sin comillas, porque es una variable y no un nombre de columna).

In [ ]:
# COMPLETE AQUÍ: reemplace cada ___
secop_anonimo = secop.drop(columns=___)
print("Columnas antes:", secop.shape[1], "→ después:", secop_anonimo.shape[1])

#### Documentación para consultar

Términos usados en esta sección. Consulte los enlaces para ampliar.

| Término | Para qué sirve | Enlace |
|---|---|---|
| `requests.get()` | Consulta una dirección web (API) desde Python; `params` agrega los parámetros de la consulta | [Ver documentación](https://requests.readthedocs.io/en/latest/user/quickstart/) |
| `try` / `except` | Manejo de errores: intentar algo y reaccionar si falla | [Ver documentación](https://docs.python.org/es/3/tutorial/errors.html) |
| Módulo `json` | Lee (`json.load`) y escribe (`json.dump`, `json.dumps`) datos en formato JSON | [Ver documentación](https://docs.python.org/es/3/library/json.html) |
| Diccionarios | Estructuras clave → valor, como los registros JSON | [Ver documentación](https://docs.python.org/es/3/tutorial/datastructures.html#dictionaries) |
| API de datos.gov.co (Socrata) | Cómo funcionan las direcciones (*endpoints*) de la API de datos abiertos | [Ver documentación](https://dev.socrata.com/docs/endpoints.html) |
| Parámetro `$limit` | Cuántos registros devuelve la API en cada consulta | [Ver documentación](https://dev.socrata.com/docs/queries/limit.html) |
| Portal datos.gov.co | Catálogo de datos abiertos de Colombia | [Ver documentación](https://www.datos.gov.co) |
| `pd.json_normalize()` | Aplana registros JSON (con campos anidados) y los convierte en tabla | [Ver documentación](https://pandas.pydata.org/docs/reference/api/pandas.json_normalize.html) |
| `.filter()` | Selecciona columnas cuyo nombre contiene un texto (`like=`) | [Ver documentación](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.filter.html) |
| `.dtypes` | Tipo de dato de cada columna (número, texto, fecha…) | [Ver documentación](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.dtypes.html) |
| `pd.to_numeric()` | Convierte texto a número; `errors="coerce"` deja vacío lo que no se puede convertir | [Ver documentación](https://pandas.pydata.org/docs/reference/api/pandas.to_numeric.html) |
| `.groupby()` | Agrupa las filas por los valores de una columna para calcular algo por grupo | [Ver documentación](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html) |
| `sum`, `mean`, `count`, `nunique`, `agg` | Cálculos que se aplican después de `groupby`: suma, promedio, conteo, conteo de valores distintos y varios cálculos a la vez | [Ver documentación](https://pandas.pydata.org/docs/reference/groupby.html) |
| `.drop()` | Elimina columnas (`columns=`) o filas de una tabla | [Ver documentación](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop.html) |
| Ley 1581 de 2012 | Ley colombiana de protección de datos personales | [Ver documentación](http://www.secretariasenado.gov.co/senado/basedoc/ley_1581_2012.html) |

**Pregunta 2.1.** ¿En qué se diferencia este dato del CSV de pedidos? ¿Por qué se llama *semiestructurado*?

**Pregunta 2.2.** ¿Para qué le serviría esta fuente externa a una empresa colombiana? Dé un ejemplo concreto.

**Pregunta 2.3.** ¿Qué columnas personales encontró? ¿Debería una empresa almacenarlas si no las necesita?

*Respuesta:* 

### Reto opcional: cace su propia fuente

1. Entre a [datos.gov.co](https://www.datos.gov.co) y busque un conjunto de su interés (salud, movilidad, educación, clima…).
2. En la página del conjunto, el identificador aparece en la dirección con la forma `xxxx-xxxx` (también en el botón **API**).
3. Cambie `ID_DATASET` en la celda de descarga y vuelva a ejecutar esta sección.


---
## 3. Fuente 3 · Datos no estructurados: lo que escriben los clientes

Las reseñas de Olist incluyen un **comentario en texto libre**, escrito por personas y en **portugués**. No tiene columnas ni formato: es el tipo de dato que más abunda y el que menos se aprovecha.

In [ ]:
resenas = pd.read_csv(f"{CARPETA}/olist_order_reviews_dataset.csv")
con_texto = resenas["review_comment_message"].notna().mean()
print(f"Reseñas totales: {len(resenas):,}")
print(f"Reseñas con comentario escrito: {con_texto:.1%}")
resenas.loc[resenas["review_comment_message"].notna(), ["review_score", "review_comment_message"]].head()

Para convertir texto en información contamos palabras, quitando las **palabras vacías** (*stopwords*): artículos, preposiciones y conectores que no aportan significado.

In [ ]:
STOP_PT = set("""
a o e é as os de da do das dos em no na nos nas um uma uns umas que se por para pra com sem
não nao mais muito muita mas foi ser ter tem está esta estou isso esse essa este aqui já ja
ainda até ate meu minha me eu ele ela eles elas você voce pois porque como quando qual sobre
ao aos pelo pela todo toda tudo bem só so ou nem lhe era vai vou sim fiz dia dias
""".split())

def palabras_frecuentes(textos, n=15):
    """Cuenta las palabras más frecuentes de una serie de textos."""
    palabras = (textos.dropna()
                      .str.lower()
                      .str.replace(r"[^a-záéíóúâêôãõçà\s]", " ", regex=True)
                      .str.split()
                      .explode())
    palabras = palabras[(palabras.str.len() >= 3) & (~palabras.isin(STOP_PT))]
    return palabras.value_counts().head(n)

In [ ]:
malas = palabras_frecuentes(resenas.loc[resenas["review_score"] == 1, "review_comment_message"])
malas.sort_values().plot(kind="barh", color="indianred", title="Palabras más frecuentes en reseñas de 1 estrella")
plt.show()

**COMPLETE AQUÍ**: haga lo mismo con las reseñas de **5 estrellas**. El espacio es un número, sin comillas.

In [ ]:
# COMPLETE AQUÍ: reemplace cada ___
buenas = palabras_frecuentes(resenas.loc[resenas["review_score"] == ___, "review_comment_message"])
buenas.sort_values().plot(kind="barh", color="seagreen", title="Palabras más frecuentes en reseñas de 5 estrellas")
plt.show()

**Mini glosario portugués → español**

| Portugués | Español | Portugués | Español |
|---|---|---|---|
| produto | producto | chegou | llegó |
| recebi | recibí | entrega / entregue | entrega / entregado |
| prazo | plazo | antes do prazo | antes del plazo |
| ainda | todavía | nunca | nunca |
| ótimo | excelente | recomendo | recomiendo |
| compra | compra | loja | tienda |


Ahora conectemos con la semana 6: ¿cuántas reseñas **mencionan la entrega**?

**COMPLETE AQUÍ**: use el patrón `"entreg|cheg|receb|prazo"` para detectar menciones a la entrega.

In [ ]:
# COMPLETE AQUÍ: reemplace cada ___
resenas["menciona_entrega"] = resenas["review_comment_message"].str.contains("___", case=False, na=False)

(resenas[resenas["review_comment_message"].notna()]
    .groupby("review_score")["menciona_entrega"].mean()
    .mul(100)
    .plot(kind="bar", title="% de comentarios que mencionan la entrega, por calificación"))
plt.ylabel("%")
plt.show()

#### Documentación para consultar

Términos usados en esta sección. Consulte los enlaces para ampliar.

| Término | Para qué sirve | Enlace |
|---|---|---|
| `.str` | Accesor para aplicar operaciones de texto a toda una columna | [Ver documentación](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.html) |
| `.str.lower()` | Convierte el texto a minúsculas | [Ver documentación](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.lower.html) |
| `.str.replace()` | Reemplaza partes del texto; con `regex=True` acepta expresiones regulares | [Ver documentación](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.replace.html) |
| Módulo `re` (expresiones regulares) | Sintaxis de patrones como `[^a-z]` (cualquier carácter que no sea letra) o la barra vertical ("o") | [Ver documentación](https://docs.python.org/es/3/library/re.html) |
| `.str.split()` | Separa el texto en una lista de palabras | [Ver documentación](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.split.html) |
| `.explode()` | Convierte cada elemento de una lista en una fila propia | [Ver documentación](https://pandas.pydata.org/docs/reference/api/pandas.Series.explode.html) |
| `.str.len()` | Longitud de cada texto | [Ver documentación](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.len.html) |
| `.isin()` | Verifica si cada valor pertenece a una lista o conjunto | [Ver documentación](https://pandas.pydata.org/docs/reference/api/pandas.Series.isin.html) |
| `.dropna()` | Elimina los valores vacíos | [Ver documentación](https://pandas.pydata.org/docs/reference/api/pandas.Series.dropna.html) |
| `.str.contains()` | Verifica si el texto contiene un patrón; la barra vertical del patrón significa "o" | [Ver documentación](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.contains.html) |
| Guía: trabajar con texto | Todas las operaciones de texto en pandas | [Ver documentación](https://pandas.pydata.org/docs/user_guide/text.html) |

**Pregunta 3.1.** ¿De qué se quejan los clientes insatisfechos? ¿Qué elogian los satisfechos? ¿Coincide con lo que encontró la semana pasada sobre las entregas?

**Pregunta 3.2.** ¿Qué parte de esta información **no** podría obtenerse de las tablas estructuradas?

*Respuesta:* 

---
## 4. Volumen: ¿cuándo pandas ya no alcanza?

pandas carga **todos los datos en la memoria RAM** del computador. Eso funciona bien hasta cierto tamaño. Vamos a medirlo con la tabla más grande de Olist: **geolocalización**, que tiene alrededor de un millón de filas.

In [ ]:
ruta_geo = f"{CARPETA}/olist_geolocation_dataset.csv"
geo = pd.read_csv(ruta_geo)

memoria = pd.Series({
    "pedidos": pedidos.memory_usage(deep=True).sum(),
    "reseñas": resenas.memory_usage(deep=True).sum(),
    "SECOP (muestra)": secop.memory_usage(deep=True).sum(),
    "geolocalización": geo.memory_usage(deep=True).sum(),
}) / 1e6

print(f"Filas en geolocalización: {len(geo):,}")
memoria.round(1).rename("MB en memoria").to_frame()

### Formatos: CSV frente a Parquet

**Parquet** es un formato **columnar y comprimido** muy usado en Big Data (Spark, Hadoop, data lakes). Comparemos tamaño en disco y velocidad de lectura.

In [ ]:
geo.to_parquet("geolocalizacion.parquet", index=False)

inicio = time.perf_counter(); pd.read_csv(ruta_geo);                       t_csv = time.perf_counter() - inicio
inicio = time.perf_counter(); pd.read_parquet("geolocalizacion.parquet"); t_parquet = time.perf_counter() - inicio

comparacion = pd.DataFrame({
    "Tamaño en disco (MB)": [os.path.getsize(ruta_geo) / 1e6, os.path.getsize("geolocalizacion.parquet") / 1e6],
    "Tiempo de lectura (s)": [t_csv, t_parquet],
}, index=["CSV", "Parquet"])
comparacion.round(2)

### Procesar por bloques

Si un archivo no cabe en memoria, se puede leer **por partes** (`chunksize`) y acumular resultados. Es la idea que, llevada a muchas máquinas en paralelo, da origen a herramientas como **Hadoop MapReduce** y **Spark**.

**COMPLETE AQUÍ**: lea el archivo en bloques de **100 000** filas. En Python se escribe `100000` o `100_000`, sin puntos ni espacios.

In [ ]:
# COMPLETE AQUÍ: reemplace cada ___
conteo_estados = pd.Series(dtype="float64")
bloques = 0

for bloque in pd.read_csv(ruta_geo, chunksize=___, usecols=["geolocation_state"]):
    conteo_estados = conteo_estados.add(bloque["geolocation_state"].value_counts(), fill_value=0)
    bloques += 1

print(f"Se procesaron {bloques} bloques")
conteo_estados.sort_values(ascending=False).head(5).astype(int)

### Experimento mental: ¿y si la empresa creciera?

In [ ]:
bytes_por_fila = geo.memory_usage(deep=True).sum() / len(geo)

try:
    import psutil
    ram_gb = psutil.virtual_memory().total / 1e9
except ImportError:
    ram_gb = 12.7   # RAM aproximada de una sesión gratuita de Colab

print(f"Memoria por fila: {bytes_por_fila:,.0f} bytes")
print(f"RAM de esta máquina: {ram_gb:,.1f} GB\n")
for filas in [1e6, 1e7, 1e8, 5e8, 1e9]:
    gb = bytes_por_fila * filas / 1e9
    estado = "cabe" if gb < ram_gb * 0.5 else "NO cabe"
    print(f"{filas:>15,.0f} filas → {gb:>8,.1f} GB  {estado}")

#### Documentación para consultar

Términos usados en esta sección. Consulte los enlaces para ampliar.

| Término | Para qué sirve | Enlace |
|---|---|---|
| `.memory_usage()` | Memoria que ocupa cada columna; con `deep=True` mide también el texto | [Ver documentación](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.memory_usage.html) |
| `.to_parquet()` | Guarda una tabla en formato Parquet | [Ver documentación](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_parquet.html) |
| `pd.read_parquet()` | Lee un archivo Parquet | [Ver documentación](https://pandas.pydata.org/docs/reference/api/pandas.read_parquet.html) |
| Apache Parquet | Documentación del formato columnar usado en Big Data | [Ver documentación](https://parquet.apache.org/docs/overview/) |
| `time.perf_counter()` | Reloj de alta precisión para medir cuánto tarda una operación | [Ver documentación](https://docs.python.org/es/3/library/time.html#time.perf_counter) |
| `os.path.getsize()` | Tamaño de un archivo en bytes | [Ver documentación](https://docs.python.org/es/3/library/os.path.html#os.path.getsize) |
| Lectura por bloques (`chunksize`) | Cómo recorrer un archivo grande por partes sin cargarlo completo | [Ver documentación](https://pandas.pydata.org/docs/user_guide/io.html#iterating-through-files-chunk-by-chunk) |
| `.add()` | Suma dos series alineando por etiqueta; `fill_value=0` trata las etiquetas faltantes como cero | [Ver documentación](https://pandas.pydata.org/docs/reference/api/pandas.Series.add.html) |
| `psutil` | Información del computador, como la memoria RAM total | [Ver documentación](https://psutil.readthedocs.io/) |
| Guía: escalar a grandes volúmenes | Qué hacer cuando los datos no caben en memoria | [Ver documentación](https://pandas.pydata.org/docs/user_guide/scale.html) |

**Pregunta 4.1.** ¿Cuántas filas soportaría este computador antes de quedarse sin memoria? Si Olist registrara la ubicación GPS de cada repartidor cada 10 segundos (fuente M2M), ¿cuánto tardaría en llegar a ese límite?

**Pregunta 4.2.** ¿Qué ventajas mostró Parquet frente a CSV? ¿Qué relación tiene esto con las V de **volumen** y **velocidad**?

*Respuesta:* 

---
## 5. Inventario de fuentes

Esta celda reúne las cifras de las fuentes que trabajó hoy.

In [ ]:
inventario = pd.DataFrame([
    {"Fuente": "Pedidos Olist",       "Estructura": "Estructurado",     "Filas": len(pedidos), "Columnas": pedidos.shape[1]},
    {"Fuente": "SECOP II (API)",      "Estructura": "Semiestructurado", "Filas": len(secop),   "Columnas": secop.shape[1]},
    {"Fuente": "Comentarios reseñas", "Estructura": "No estructurado",  "Filas": int(resenas["review_comment_message"].notna().sum()), "Columnas": 1},
    {"Fuente": "Geolocalización",     "Estructura": "Estructurado",     "Filas": len(geo),     "Columnas": geo.shape[1]},
])
inventario["MB en memoria"] = [memoria["pedidos"], memoria["SECOP (muestra)"], memoria["reseñas"], memoria["geolocalización"]]
inventario["MB en memoria"] = inventario["MB en memoria"].round(1)
inventario

Complete la tabla con su análisis:

| Fuente | Categoría según Joyanes | Estructura | Interna / externa | V de Big Data más evidente | Valor para el negocio (alto / medio / bajo y por qué) |
|---|---|---|---|---|---|
| Pedidos Olist | | Estructurado | | | |
| SECOP II (API) | | Semiestructurado | | | |
| Comentarios de reseñas | | No estructurado | | | |
| Geolocalización | | Estructurado | | | |


### Fuentes que la empresa **no** está usando

Proponga **al menos tres** fuentes adicionales que Olist (o su empresa caso) podría aprovechar.

| Fuente propuesta | Categoría según Joyanes | Estructura | ¿Qué pregunta de negocio ayudaría a responder? |
|---|---|---|---|
| | | | |
| | | | |
| | | | |


### Conclusión

**Pregunta final.** De todas las fuentes analizadas y propuestas, ¿cuál aporta **más valor** al negocio y por qué? Justifique con las V de Big Data (volumen, velocidad, variedad, veracidad, valor). Máximo 200 palabras.

*Respuesta:* 

---
## Documentación de referencia

Todos los términos usados en este notebook, agrupados por tema. La documentación oficial de pandas está en inglés: si lo prefiere, use la opción **Traducir** del navegador (clic derecho sobre la página). La documentación de Python enlazada está en español.

**Guías generales**

| Término | Para qué sirve | Enlace |
|---|---|---|
| Guía: datos faltantes | Cómo detectar y tratar valores vacíos | [Ver documentación](https://pandas.pydata.org/docs/user_guide/missing_data.html) |
| Tutoriales de introducción a pandas | Diez tutoriales cortos: leer datos, seleccionar, graficar, combinar tablas, fechas, texto | [Ver documentación](https://pandas.pydata.org/docs/getting_started/intro_tutorials/index.html) |
| 10 minutos con pandas | Recorrido rápido por las funciones principales | [Ver documentación](https://pandas.pydata.org/docs/user_guide/10min.html) |

**Lectura y exploración**

| Término | Para qué sirve | Enlace |
|---|---|---|
| `pd.read_csv()` | Lee un archivo CSV y lo convierte en tabla (DataFrame). `parse_dates` indica qué columnas son fechas y `chunksize` permite leer por bloques | [Ver documentación](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) |
| `.info()` | Resumen de la tabla: columnas, tipos de dato y valores no vacíos | [Ver documentación](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.info.html) |
| `.dtypes` | Tipo de dato de cada columna (número, texto, fecha…) | [Ver documentación](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.dtypes.html) |

**Selección y filtrado**

| Término | Para qué sirve | Enlace |
|---|---|---|
| `.isna()` | Devuelve `True` donde el valor está vacío (NaN) | [Ver documentación](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.isna.html) |
| `.filter()` | Selecciona columnas cuyo nombre contiene un texto (`like=`) | [Ver documentación](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.filter.html) |
| `.drop()` | Elimina columnas (`columns=`) o filas de una tabla | [Ver documentación](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop.html) |
| `.isin()` | Verifica si cada valor pertenece a una lista o conjunto | [Ver documentación](https://pandas.pydata.org/docs/reference/api/pandas.Series.isin.html) |
| `.dropna()` | Elimina los valores vacíos | [Ver documentación](https://pandas.pydata.org/docs/reference/api/pandas.Series.dropna.html) |

**Agrupar y resumir**

| Término | Para qué sirve | Enlace |
|---|---|---|
| `.mean()` | Promedio. Sobre una columna de `True`/`False` da la proporción de `True` | [Ver documentación](https://pandas.pydata.org/docs/reference/api/pandas.Series.mean.html) |
| `.mul()` / `.round()` | Multiplica (por ejemplo, por 100 para obtener porcentajes) y redondea | [Ver documentación](https://pandas.pydata.org/docs/reference/api/pandas.Series.mul.html) |
| `.value_counts()` | Cuenta cuántas veces aparece cada valor. Con `normalize=True` da proporciones | [Ver documentación](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html) |
| `.groupby()` | Agrupa las filas por los valores de una columna para calcular algo por grupo | [Ver documentación](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html) |
| `sum`, `mean`, `count`, `nunique`, `agg` | Cálculos que se aplican después de `groupby`: suma, promedio, conteo, conteo de valores distintos y varios cálculos a la vez | [Ver documentación](https://pandas.pydata.org/docs/reference/groupby.html) |
| `.add()` | Suma dos series alineando por etiqueta; `fill_value=0` trata las etiquetas faltantes como cero | [Ver documentación](https://pandas.pydata.org/docs/reference/api/pandas.Series.add.html) |

**Texto**

| Término | Para qué sirve | Enlace |
|---|---|---|
| `.str` | Accesor para aplicar operaciones de texto a toda una columna | [Ver documentación](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.html) |
| `.str.lower()` | Convierte el texto a minúsculas | [Ver documentación](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.lower.html) |
| `.str.replace()` | Reemplaza partes del texto; con `regex=True` acepta expresiones regulares | [Ver documentación](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.replace.html) |
| Módulo `re` (expresiones regulares) | Sintaxis de patrones como `[^a-z]` (cualquier carácter que no sea letra) o la barra vertical ("o") | [Ver documentación](https://docs.python.org/es/3/library/re.html) |
| `.str.split()` | Separa el texto en una lista de palabras | [Ver documentación](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.split.html) |
| `.explode()` | Convierte cada elemento de una lista en una fila propia | [Ver documentación](https://pandas.pydata.org/docs/reference/api/pandas.Series.explode.html) |
| `.str.len()` | Longitud de cada texto | [Ver documentación](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.len.html) |
| `.str.contains()` | Verifica si el texto contiene un patrón; la barra vertical del patrón significa "o" | [Ver documentación](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.contains.html) |
| Guía: trabajar con texto | Todas las operaciones de texto en pandas | [Ver documentación](https://pandas.pydata.org/docs/user_guide/text.html) |

**Datos semiestructurados y APIs**

| Término | Para qué sirve | Enlace |
|---|---|---|
| `requests.get()` | Consulta una dirección web (API) desde Python; `params` agrega los parámetros de la consulta | [Ver documentación](https://requests.readthedocs.io/en/latest/user/quickstart/) |
| Módulo `json` | Lee (`json.load`) y escribe (`json.dump`, `json.dumps`) datos en formato JSON | [Ver documentación](https://docs.python.org/es/3/library/json.html) |
| API de datos.gov.co (Socrata) | Cómo funcionan las direcciones (*endpoints*) de la API de datos abiertos | [Ver documentación](https://dev.socrata.com/docs/endpoints.html) |
| Parámetro `$limit` | Cuántos registros devuelve la API en cada consulta | [Ver documentación](https://dev.socrata.com/docs/queries/limit.html) |
| Portal datos.gov.co | Catálogo de datos abiertos de Colombia | [Ver documentación](https://www.datos.gov.co) |
| `pd.json_normalize()` | Aplana registros JSON (con campos anidados) y los convierte en tabla | [Ver documentación](https://pandas.pydata.org/docs/reference/api/pandas.json_normalize.html) |
| `pd.to_numeric()` | Convierte texto a número; `errors="coerce"` deja vacío lo que no se puede convertir | [Ver documentación](https://pandas.pydata.org/docs/reference/api/pandas.to_numeric.html) |
| Ley 1581 de 2012 | Ley colombiana de protección de datos personales | [Ver documentación](http://www.secretariasenado.gov.co/senado/basedoc/ley_1581_2012.html) |

**Volumen y formatos**

| Término | Para qué sirve | Enlace |
|---|---|---|
| `.memory_usage()` | Memoria que ocupa cada columna; con `deep=True` mide también el texto | [Ver documentación](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.memory_usage.html) |
| `.to_parquet()` | Guarda una tabla en formato Parquet | [Ver documentación](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_parquet.html) |
| `pd.read_parquet()` | Lee un archivo Parquet | [Ver documentación](https://pandas.pydata.org/docs/reference/api/pandas.read_parquet.html) |
| Apache Parquet | Documentación del formato columnar usado en Big Data | [Ver documentación](https://parquet.apache.org/docs/overview/) |
| `time.perf_counter()` | Reloj de alta precisión para medir cuánto tarda una operación | [Ver documentación](https://docs.python.org/es/3/library/time.html#time.perf_counter) |
| `os.path.getsize()` | Tamaño de un archivo en bytes | [Ver documentación](https://docs.python.org/es/3/library/os.path.html#os.path.getsize) |
| Lectura por bloques (`chunksize`) | Cómo recorrer un archivo grande por partes sin cargarlo completo | [Ver documentación](https://pandas.pydata.org/docs/user_guide/io.html#iterating-through-files-chunk-by-chunk) |
| `psutil` | Información del computador, como la memoria RAM total | [Ver documentación](https://psutil.readthedocs.io/) |
| Guía: escalar a grandes volúmenes | Qué hacer cuando los datos no caben en memoria | [Ver documentación](https://pandas.pydata.org/docs/user_guide/scale.html) |

**Python básico**

| Término | Para qué sirve | Enlace |
|---|---|---|
| `try` / `except` | Manejo de errores: intentar algo y reaccionar si falla | [Ver documentación](https://docs.python.org/es/3/tutorial/errors.html) |
| Diccionarios | Estructuras clave → valor, como los registros JSON | [Ver documentación](https://docs.python.org/es/3/tutorial/datastructures.html#dictionaries) |


---
## Entregable de la semana 7

- [ ] Notebook con las tres fuentes cargadas y todas las celdas COMPLETE AQUÍ completadas y ejecutadas sin errores.
- [ ] Tabla de categorías de Joyanes con ejemplos propios.
- [ ] Preguntas respondidas.
- [ ] Inventario de fuentes y tabla de fuentes no aprovechadas.
- [ ] Conclusión sobre la fuente de mayor valor.

Descárguelo con **Archivo → Descargar → Descargar .ipynb** y súbalo a la plataforma del curso.
